# 🔬 Spectral Analysis Pipeline
**Raman / IR — Hypothesis-driven, interactive**

### Steps
1. Mount Google Drive & install packages
2. State your hypothesis
3. Load your Excel file
4. Normalise (Peak · SNV · MSC) — all three saved as Excel
5. Choose normalisation for analysis
6. Select classes to compare (2 or more)
7. Select which figures to generate (or generate all)
8. Full statistical analysis → publishable figures + tables
9. AI Hypothesis Interpretation (Claude API + latest research)
10. Download results as ZIP

---
▶ **Run cells top to bottom. Each cell will prompt you for input where needed.**

## Cell 1 — Mount Google Drive & install packages

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✓ Google Drive mounted at /content/drive')

!pip install --quiet openpyxl anthropic
print('✓ Packages ready')

## Cell 2 — Pipeline code (run once, do not edit)

In [ ]:
import os, textwrap, warnings
from datetime import datetime
from itertools import combinations

import numpy as np
import pandas as pd

# Fix #7: Use inline backend on Colab instead of global Agg which suppresses display
try:
    import google.colab
    _IN_COLAB = True
    import matplotlib
    matplotlib.use('module://matplotlib_inline.backend_inline')
except ImportError:
    _IN_COLAB = False
    import matplotlib
    matplotlib.use('Agg')

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from scipy.stats import f_oneway
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cross_decomposition import PLSRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import cross_val_score, StratifiedKFold
from IPython.display import display, Image as IPImage

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.family': 'serif', 'font.size': 11,
    'axes.titlesize': 13, 'axes.labelsize': 11,
    'xtick.labelsize': 9, 'ytick.labelsize': 9,
    'legend.fontsize': 10, 'figure.dpi': 150,
    'savefig.dpi': 300, 'savefig.bbox': 'tight',
    'axes.spines.top': False, 'axes.spines.right': False,
})

PALETTE = ['#2166AC','#D6604D','#4DAC26','#7B2D8B',
           '#F4A582','#1A9850','#E08214','#762A83']

def color_for(label, labels):
    return PALETTE[labels.index(label) % len(PALETTE)]

def savefig(fig, path_png):
    fig.savefig(path_png)
    fig.savefig(path_png.replace('.png', '.svg'))
    if _IN_COLAB:
        display(IPImage(path_png))
    plt.close(fig)
    print(f'  ✓  {os.path.basename(path_png)}  +  .svg')

def ci95(data):
    n = data.shape[0]
    se = data.std(axis=0, ddof=1) / np.sqrt(n)
    return stats.t.ppf(0.975, df=n-1) * se

def sig_stars(p):
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return 'ns'

# ── Normalisations ─────────────────────────────────────────────────────────
def normalise_peak(X):
    mx = np.max(np.abs(X), axis=1, keepdims=True); mx[mx==0]=1
    return X / mx

def normalise_snv(X):
    mu = X.mean(axis=1, keepdims=True)
    sig = X.std(axis=1, keepdims=True); sig[sig==0]=1
    return (X - mu) / sig

def normalise_msc(X):
    ref = X.mean(axis=0); out = np.zeros_like(X)
    for i, row in enumerate(X):
        slope, intercept, *_ = stats.linregress(ref, row)
        if slope == 0: slope = 1
        out[i] = (row - intercept) / slope
    return out

NORM_FNS = {'peak': normalise_peak, 'snv': normalise_snv, 'msc': normalise_msc}

# Fix #4: Cache so switching norm doesn't re-run everything
_norm_cache = {}

def get_normalised(name, spectra):
    if name not in _norm_cache:
        print(f'  Computing {name.upper()} normalisation...', end=' ', flush=True)
        _norm_cache[name] = NORM_FNS[name](spectra)
        print('done.')
    else:
        print(f'  Using cached {name.upper()} normalisation.')
    return _norm_cache[name]

# ── Figures ────────────────────────────────────────────────────────────────
def fig_mean_spectra(wn, groups, norm_name, out):
    labels = [g[0] for g in groups]
    fig, ax = plt.subplots(figsize=(9,4))
    for lbl, X in groups:
        m=X.mean(axis=0); ci=ci95(X); c=color_for(lbl,labels)
        ax.plot(wn,m,color=c,lw=1.5,label=lbl)
        ax.fill_between(wn,m-ci,m+ci,color=c,alpha=0.18)
    ax.set_xlabel('Wavenumber (cm\u207b\u00b9)'); ax.set_ylabel(f'{norm_name} Intensity')
    ax.set_title(f'Mean Spectra \u00b1 95 % CI  [{norm_name}]')
    ax.legend(frameon=False,bbox_to_anchor=(1,1),loc='upper left')
    fig.tight_layout(); savefig(fig,out)

def fig_pca(wn, groups, out_scores, out_loadings):
    labels=[g[0] for g in groups]
    X=np.vstack([g[1] for g in groups])
    y=np.concatenate([[lbl]*len(mx) for lbl,mx in groups])
    Xs=StandardScaler().fit_transform(X)
    n_comp=min(5,X.shape[0]-1,X.shape[1])
    pca=PCA(n_components=n_comp); sc=pca.fit_transform(Xs); ev=pca.explained_variance_ratio_*100
    fig,ax=plt.subplots(figsize=(7,6))
    for lbl in labels:
        m=y==lbl; ax.scatter(sc[m,0],sc[m,1],c=color_for(lbl,labels),s=35,alpha=0.75,edgecolors='white',lw=0.3,label=lbl)
    ax.axhline(0,color='grey',lw=0.5,ls='--'); ax.axvline(0,color='grey',lw=0.5,ls='--')
    ax.set_xlabel(f'PC 1 ({ev[0]:.1f} %)'); ax.set_ylabel(f'PC 2 ({ev[1]:.1f} %)')
    ax.set_title('PCA Scores'); ax.legend(frameon=False,bbox_to_anchor=(1,1),loc='upper left')
    fig.tight_layout(); savefig(fig,out_scores)
    n_show=min(3,n_comp)
    fig,axes=plt.subplots(n_show,1,figsize=(9,2.5*n_show),sharex=True)
    if n_show==1: axes=[axes]
    for i,ax in enumerate(axes):
        ax.plot(wn,pca.components_[i],color=PALETTE[i],lw=1)
        ax.axhline(0,color='grey',lw=0.5,ls='--'); ax.set_ylabel(f'PC {i+1} loading')
    axes[-1].set_xlabel('Wavenumber (cm\u207b\u00b9)'); axes[0].set_title('PCA Loadings')
    fig.tight_layout(); savefig(fig,out_loadings)
    return pca,sc,ev,y

def fig_significance(wn, groups, out, p_threshold=0.05):
    labels=[g[0] for g in groups]
    if len(groups)==2:
        _,p=stats.ttest_ind(groups[0][1],groups[1][1],axis=0,equal_var=False); test_label='Welch t-test'
    else:
        p=np.array([f_oneway(*[g[1][:,j] for g in groups]).pvalue for j in range(groups[0][1].shape[1])]); test_label='One-way ANOVA'
    logp=-np.log10(np.clip(p,1e-300,1)); sig=p<p_threshold
    fig,axes=plt.subplots(2,1,figsize=(9,6),sharex=True,gridspec_kw={'height_ratios':[2,1]})
    for lbl,X in groups:
        axes[0].plot(wn,X.mean(axis=0),color=color_for(lbl,labels),lw=1.2,label=lbl)
    axes[0].set_ylabel('Normalised Intensity')
    axes[0].legend(frameon=False,bbox_to_anchor=(1,1),loc='upper left')
    axes[0].set_title(f'Point-wise {test_label}')
    axes[1].fill_between(wn,0,logp,where=~sig,color='lightgrey',step='mid',label=f'p \u2265 {p_threshold}')
    axes[1].fill_between(wn,0,logp,where=sig,color='#D6604D',alpha=0.85,step='mid',label=f'p < {p_threshold}')
    axes[1].axhline(-np.log10(p_threshold),color='black',lw=0.8,ls='--',label=f'\u03b1 = {p_threshold}')
    axes[1].set_ylabel('\u2212log\u2081\u2080(p)'); axes[1].set_xlabel('Wavenumber (cm\u207b\u00b9)')
    axes[1].legend(frameon=False,fontsize=9,bbox_to_anchor=(1,1),loc='upper left')
    fig.tight_layout(); savefig(fig,out)
    return p

def fig_heatmap(wn, groups, out):
    labels=[g[0] for g in groups]; chunks,dividers=[],[]; cursor=0
    for lbl,X in groups:
        n_show=min(len(X),max(10,80//len(groups))); step=max(1,len(X)//n_show)
        chunks.append(X[::step]); cursor+=len(X[::step]); dividers.append(cursor)
    Xplot=np.vstack(chunks)
    fig,ax=plt.subplots(figsize=(11,max(4,len(Xplot)*0.12+1.5)))
    im=ax.imshow(Xplot,aspect='auto',cmap='RdYlBu_r',extent=[wn[0],wn[-1],len(Xplot),0])
    fig.colorbar(im,ax=ax,pad=0.02,shrink=0.75).set_label('Normalised Intensity')
    for d in dividers[:-1]: ax.axhline(d,color='white',lw=1.5,ls='--')
    patches=[mpatches.Patch(color=color_for(lbl,labels),label=lbl) for lbl in labels]
    ax.legend(handles=patches,loc='upper right',frameon=False,fontsize=9)
    ax.set_xlabel('Wavenumber (cm\u207b\u00b9)'); ax.set_ylabel('Sample index'); ax.set_title('Spectral Heatmap by Class')
    fig.tight_layout(); savefig(fig,out)

def fig_violin(wn, groups, pca_sc, pca_ev, y, out):
    labels=[g[0] for g in groups]; n_pc=min(3,pca_sc.shape[1])
    fig,axes=plt.subplots(1,n_pc,figsize=(4.5*n_pc,5))
    if n_pc==1: axes=[axes]
    for i,ax in enumerate(axes):
        data_list=[pca_sc[y==lbl,i] for lbl in labels]; positions=list(range(1,len(labels)+1))
        parts=ax.violinplot(data_list,positions=positions,showmedians=False,showextrema=False)
        for pc,lbl in zip(parts['bodies'],labels): pc.set_facecolor(color_for(lbl,labels)); pc.set_alpha(0.55)
        ax.boxplot(data_list,positions=positions,widths=0.1,
                   medianprops=dict(color='black',lw=2),whiskerprops=dict(color='grey'),
                   capprops=dict(color='grey'),flierprops=dict(marker='o',ms=3,color='grey',alpha=0.5))
        ax.set_xticks(positions); ax.set_xticklabels(labels,rotation=20,ha='right')
        ax.set_ylabel(f'PC {i+1} score ({pca_ev[i]:.1f} %)')
        if len(labels)==2:
            _,pv=stats.ttest_ind(data_list[0],data_list[1])
            ymax=max(d.max() for d in data_list); ax.annotate(sig_stars(pv),xy=(1.5,ymax*1.08),ha='center',fontsize=14)
        else:
            pv=f_oneway(*data_list).pvalue; ax.set_title(f'ANOVA {sig_stars(pv)}',fontsize=10)
    fig.suptitle('PC Score Distributions',y=1.01,fontsize=12)
    fig.tight_layout(); savefig(fig,out)

def fig_lda(groups, out):
    labels=[g[0] for g in groups]
    X=np.vstack([g[1] for g in groups]); y=np.concatenate([[i]*len(g[1]) for i,g in enumerate(groups)])
    n_comp=min(X.shape[0]-1,X.shape[1],50)
    Xr=PCA(n_components=n_comp).fit_transform(StandardScaler().fit_transform(X))
    lda=LinearDiscriminantAnalysis(); lda.fit(Xr,y); sc=lda.transform(Xr)
    cv=StratifiedKFold(n_splits=min(5,min(len(g[1]) for g in groups)))
    acc=cross_val_score(lda,Xr,y,cv=cv,scoring='accuracy').mean()
    fig,ax=plt.subplots(figsize=(7,6))
    if sc.shape[1]>=2:
        for i,lbl in enumerate(labels):
            m=y==i; ax.scatter(sc[m,0],sc[m,1],c=color_for(lbl,labels),s=35,alpha=0.75,edgecolors='white',lw=0.3,label=lbl)
        ax.set_xlabel('LD 1'); ax.set_ylabel('LD 2')
    else:
        for i,lbl in enumerate(labels):
            m=y==i; ax.hist(sc[m,0],bins=20,color=color_for(lbl,labels),alpha=0.6,label=lbl,edgecolor='white')
        ax.set_xlabel('LD 1 score'); ax.set_ylabel('Count')
    ax.set_title(f'LDA  (CV accuracy: {acc:.1%})')
    ax.legend(frameon=False,bbox_to_anchor=(1,1),loc='upper left')
    fig.tight_layout(); savefig(fig,out)
    return acc

def fig_plsda(groups, out):
    labels=[g[0] for g in groups]
    X=np.vstack([g[1] for g in groups]); y=np.concatenate([[i]*len(g[1]) for i,g in enumerate(groups)]).astype(float)
    Xs=StandardScaler().fit_transform(X)
    pls=PLSRegression(n_components=2,scale=False); pls.fit(Xs,y); sc=pls.transform(Xs)
    var_exp=[np.var(Xs@pls.x_weights_[:,k])/np.sum(np.var(Xs,axis=0))*100 for k in range(2)]
    fig,ax=plt.subplots(figsize=(7,6))
    for i,lbl in enumerate(labels):
        m=y==i; ax.scatter(sc[m,0],sc[m,1],c=color_for(lbl,labels),s=35,alpha=0.75,edgecolors='white',lw=0.3,label=lbl)
    ax.axhline(0,color='grey',lw=0.5,ls='--'); ax.axvline(0,color='grey',lw=0.5,ls='--')
    ax.set_xlabel(f'LV 1 ({var_exp[0]:.1f}% X var)'); ax.set_ylabel(f'LV 2 ({var_exp[1]:.1f}% X var)')
    ax.set_title('PLS-DA Scores'); ax.legend(frameon=False,bbox_to_anchor=(1,1),loc='upper left')
    fig.tight_layout(); savefig(fig,out)

def fig_correlation(wn, groups, out):
    bin_size=max(1,len(wn)//40); idx=np.arange(0,len(wn),bin_size)
    def bin_mean(X): return np.array([X[:,i:i+bin_size].mean(axis=1) for i in idx]).T
    Xbin=np.vstack([bin_mean(g[1]) for g in groups]); corr=np.corrcoef(Xbin.T)
    tick_labels=[f'{wn[i]:.0f}' for i in idx]; step=max(1,len(tick_labels)//10)
    fig,ax=plt.subplots(figsize=(8,7))
    im=ax.imshow(corr,cmap='RdBu_r',vmin=-1,vmax=1)
    fig.colorbar(im,ax=ax,shrink=0.8).set_label('Pearson r')
    ax.set_xticks(range(0,len(tick_labels),step)); ax.set_xticklabels(tick_labels[::step],rotation=45,ha='right')
    ax.set_yticks(range(0,len(tick_labels),step)); ax.set_yticklabels(tick_labels[::step])
    ax.set_title('Wavenumber Correlation Matrix')
    fig.tight_layout(); savefig(fig,out)

def fig_pairwise_diff(wn, groups, out):
    labels=[g[0] for g in groups]; pairs=list(combinations(range(len(groups)),2)); n=len(pairs)
    fig,axes=plt.subplots(n,1,figsize=(9,2.8*n),sharex=True)
    if n==1: axes=[axes]
    for ax,(i,j) in zip(axes,pairs):
        la,lb=labels[i],labels[j]; diff=groups[i][1].mean(axis=0)-groups[j][1].mean(axis=0)
        ax.plot(wn,diff,color='#2166AC',lw=1.2)
        ax.axhline(0,color='grey',lw=0.6,ls='--')
        ax.fill_between(wn,0,diff,where=diff>0,color='#2166AC',alpha=0.15)
        ax.fill_between(wn,0,diff,where=diff<0,color='#D6604D',alpha=0.15)
        ax.set_ylabel(f'\u0394 ({la} \u2212 {lb})',fontsize=9)
    axes[-1].set_xlabel('Wavenumber (cm\u207b\u00b9)'); axes[0].set_title('Pairwise Mean Spectral Difference')
    fig.tight_layout(); savefig(fig,out)

def save_stats_table(wn, groups, p_vals, out_dir, p_threshold=0.05):
    sig_col = "Significant_p" + str(p_threshold).replace(".", "")
    rows=[]
    for j,w in enumerate(wn):
        row={'Wavenumber':round(w,2)}
        for lbl,X in groups:
            row[f'Mean_{lbl}']=round(X[:,j].mean(),6); row[f'SD_{lbl}']=round(X[:,j].std(ddof=1),6)
        row['p_value']=round(p_vals[j],6)
        row[sig_col]=bool(p_vals[j]<p_threshold)
        if len(groups)==2:
            ma,sa=groups[0][1][:,j].mean(),groups[0][1][:,j].std(ddof=1)
            mb,sb=groups[1][1][:,j].mean(),groups[1][1][:,j].std(ddof=1)
            na,nb=len(groups[0][1]),len(groups[1][1])
            pooled=np.sqrt(((na-1)*sa**2+(nb-1)*sb**2)/(na+nb-2))
            row['Cohens_d']=round((ma-mb)/pooled,4) if pooled>0 else float('nan')
        rows.append(row)
    df=pd.DataFrame(rows)
    df.to_csv(os.path.join(out_dir,'stats_summary.csv'),index=False)
    df.to_excel(os.path.join(out_dir,'stats_summary.xlsx'),index=False)
    print('  ✓  stats_summary.csv  +  .xlsx')
    return df

def save_report(out_dir, hypothesis, selected, norm_name, groups, lda_acc, p_vals, wn, p_threshold=0.05):
    # Fix #10: Richer stats with configurable p-value thresholds
    thresholds = [0.001, 0.01, p_threshold]
    n_sig_001  = (p_vals < 0.001).sum()
    n_sig_01   = (p_vals < 0.01).sum()
    n_sig      = (p_vals < p_threshold).sum()
    pct_sig    = n_sig / len(p_vals) * 100
    top_wn_idx = np.argsort(p_vals)[:10]
    lines = [
        'SPECTRAL ANALYSIS REPORT', '='*60,
        f'Date          : {datetime.now().strftime("%Y-%m-%d %H:%M")}',
        f'Normalisation : {norm_name.upper()}',
        f'P-value threshold : {p_threshold}',
        '',
        'HYPOTHESIS', '-'*60,
        textwrap.fill(hypothesis, width=70),
        '',
        'CLASSES COMPARED', '-'*60,
    ]
    for lbl, X in groups:
        lines.append(f'  {lbl:40s}  n = {len(X)}')
    lines += [
        '',
        'KEY RESULTS', '-'*60,
        f'  LDA cross-validated accuracy   : {lda_acc:.1%}',
        f'  Significant wavenumbers (p<0.001) : {n_sig_001} / {len(wn)} ({n_sig_001/len(wn)*100:.1f}%)',
        f'  Significant wavenumbers (p<0.01)  : {n_sig_01}  / {len(wn)} ({n_sig_01/len(wn)*100:.1f}%)',
        f'  Significant wavenumbers (p<{p_threshold})   : {n_sig}  / {len(wn)} ({pct_sig:.1f}%)',
        '',
        'TOP 10 MOST SIGNIFICANT WAVENUMBERS', '-'*60,
    ]
    for idx in top_wn_idx:
        lines.append(f'  {wn[idx]:.1f} cm⁻¹  p = {p_vals[idx]:.2e}  {sig_stars(p_vals[idx])}')
    rpt = '\n'.join(lines)
    with open(os.path.join(out_dir,'report.txt'),'w') as f:
        f.write(rpt)
    print('\n' + rpt)
    return {
        'n_sig': int(n_sig),
        'n_total': len(wn),
        'pct_sig': round(pct_sig, 1),
        'lda_acc': lda_acc,
        'top_wavenumbers': [(round(wn[i],1), float(p_vals[i])) for i in top_wn_idx],
    }

ALL_FIGURES = [
    '01_mean_spectra',
    '02_pca_scores',
    '03_pca_loadings',
    '04_significance',
    '05_heatmap',
    '06_violin_pc_scores',
    '07_lda',
    '08_plsda',
    '09_correlation_heatmap',
    '10_pairwise_diff',
]

def run_analysis(wn, Xn, classes, selected, norm_name, out_dir, hypothesis,
                 figures_to_run=None, p_threshold=0.05):
    # Fix #5: figure selection — None means all
    if figures_to_run is None:
        figures_to_run = set(ALL_FIGURES)
    else:
        figures_to_run = set(figures_to_run)

    groups = [(lbl, Xn[classes==lbl]) for lbl in selected]
    def out(name): return os.path.join(out_dir, name)

    pca_obj = pca_sc = pca_ev = y_arr = None
    p_vals  = None
    lda_acc = None

    print('\n--- Generating figures ---')

    if '01_mean_spectra' in figures_to_run:
        fig_mean_spectra(wn, groups, norm_name.upper(), out('01_mean_spectra.png'))

    if '02_pca_scores' in figures_to_run or '03_pca_loadings' in figures_to_run or '06_violin_pc_scores' in figures_to_run:
        pca_obj, pca_sc, pca_ev, y_arr = fig_pca(
            wn, groups,
            out('02_pca_scores.png') if '02_pca_scores' in figures_to_run else os.devnull,
            out('03_pca_loadings.png') if '03_pca_loadings' in figures_to_run else os.devnull,
        )

    if '04_significance' in figures_to_run:
        p_vals = fig_significance(wn, groups, out('04_significance.png'), p_threshold)
    else:
        # Still need p_vals for stats table & report
        if len(groups)==2:
            _, p_vals = stats.ttest_ind(groups[0][1], groups[1][1], axis=0, equal_var=False)
        else:
            p_vals = np.array([f_oneway(*[g[1][:,j] for g in groups]).pvalue
                               for j in range(groups[0][1].shape[1])])

    if '05_heatmap' in figures_to_run:
        fig_heatmap(wn, groups, out('05_heatmap.png'))

    if '06_violin_pc_scores' in figures_to_run:
        if pca_sc is None:
            pca_obj, pca_sc, pca_ev, y_arr = fig_pca(
                wn, groups, os.devnull, os.devnull)
        fig_violin(wn, groups, pca_sc, pca_ev, y_arr, out('06_violin_pc_scores.png'))

    if '07_lda' in figures_to_run:
        lda_acc = fig_lda(groups, out('07_lda.png'))

    # Fix #6: Explicit message when PLS-DA is skipped
    if '08_plsda' in figures_to_run:
        if len(selected) == 2:
            fig_plsda(groups, out('08_plsda.png'))
        else:
            print(f'  ⚠  PLS-DA skipped: requires exactly 2 classes, but {len(selected)} selected.\n'
                  f'     (Regression-based PLS-DA with a single continuous Y is not meaningful for {len(selected)} classes.)\n'
                  f'     Use LDA (figure 07) for multi-class discrimination instead.')

    if '09_correlation_heatmap' in figures_to_run:
        fig_correlation(wn, groups, out('09_correlation_heatmap.png'))

    if '10_pairwise_diff' in figures_to_run:
        fig_pairwise_diff(wn, groups, out('10_pairwise_diff.png'))

    print('\n--- Saving statistics ---')
    save_stats_table(wn, groups, p_vals, out_dir, p_threshold)
    results_summary = save_report(out_dir, hypothesis, selected, norm_name, groups,
                                   lda_acc if lda_acc is not None else float('nan'),
                                   p_vals, wn, p_threshold)

    print(f'\n✅ All outputs saved to:\n   {out_dir}')
    return results_summary

print('✓ Pipeline code loaded.')

## Cell 3 — State your hypothesis

In [ ]:
hypothesis = input('State your hypothesis: ').strip()
if not hypothesis:
    raise ValueError('Hypothesis cannot be empty. Please re-run this cell and enter your hypothesis.')
print(f'\nRecorded: "{hypothesis}"')

## Cell 4 — Load your Excel file

Your Google Drive is mounted at `/content/drive/MyDrive/`  
Example path: `/content/drive/MyDrive/my_folder/data.xlsx`

In [ ]:
filepath = input('Full path to your Excel file: ').strip().strip('"')

# Fix #3: Input validation — file existence
if not os.path.isfile(filepath):
    raise FileNotFoundError(
        f'File not found: {filepath}\n'
        f'Check the path and make sure Google Drive is mounted.')

print('Loading...')
try:
    df = pd.read_excel(filepath, header=0)
except Exception as e:
    raise RuntimeError(f'Could not read Excel file: {e}')

print(f'Shape: {df.shape[0]} samples x {df.shape[1]} columns')
print(f'\nFirst 8 columns: {list(df.columns[:8])}')
print(f'Last  5 columns: {list(df.columns[-5:])}')

non_numeric = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
print(f'\nNon-numeric columns (likely class labels): {non_numeric}')

In [ ]:
class_col = input('Name of the class/label column: ').strip()

# Fix #3: Validate the class column exists
if class_col not in df.columns:
    raise KeyError(
        f'Column "{class_col}" not found in the file.\n'
        f'Available columns: {list(df.columns[:20])}...')

spectral_cols = [c for c in df.columns
                 if c != class_col and pd.api.types.is_numeric_dtype(df[c])]

if len(spectral_cols) == 0:
    raise ValueError(
        f'No numeric spectral columns found after removing "{class_col}".\n'
        f'Make sure spectral columns are numeric (wavenumber values as column names).')

wn      = np.array([float(c) for c in spectral_cols])
spectra = df[spectral_cols].values.astype(float)
classes = df[class_col].astype(str)

print(f'\nWavenumber range : {wn.min():.1f} - {wn.max():.1f} cm⁻¹  ({len(wn)} points)')
print('\nClasses found:')
for cls in sorted(classes.unique()):
    print(f'  {cls:40s}  n = {(classes == cls).sum()}')

## Cell 5 — Normalise & save Excel files

All three methods are computed and cached. Re-running this cell is fast (uses cache).

In [ ]:
out_norm_dir = os.path.join(os.path.dirname(filepath), 'normalised')
os.makedirs(out_norm_dir, exist_ok=True)
base = os.path.splitext(os.path.basename(filepath))[0]
spectral_col_names = [str(w) for w in wn]

# Fix #4: Use get_normalised() which caches results — rerunning this cell won't redo MSC etc.
normalised = {}
for name in NORM_FNS:
    Xn = get_normalised(name, spectra)
    normalised[name] = Xn
    out_path = os.path.join(out_norm_dir, f'{base}_{name.upper()}.xlsx')
    df_out = pd.DataFrame(Xn, columns=spectral_col_names)
    df_out.insert(0, 'Class', classes.values)
    df_out.to_excel(out_path, index=False)
    print(f'  saved -> {os.path.basename(out_path)}')

print(f'\n✓ Normalised files saved to: {out_norm_dir}')

## Cell 6 — Choose normalisation & select classes

In [ ]:
print('Available: peak | snv | msc')
norm_name = input('Which normalisation to use for analysis: ').strip().lower()

# Fix #3 + #4: Validate norm name and use cache
if norm_name not in NORM_FNS:
    raise ValueError(
        f'"{norm_name}" is not a valid normalisation method.\n'
        f'Please choose one of: peak, snv, msc')

Xn = normalised[norm_name]
print(f'Using: {norm_name.upper()}')

In [ ]:
available_classes = sorted(classes.unique())
print('Available classes:')
for i, c in enumerate(available_classes, 1):
    print(f'  [{i}] {c}  (n={(classes == c).sum()})')

print('\nEnter class names separated by commas, OR enter numbers (e.g. 1,3,4).')
raw = input('Classes: ').strip()

# Fix #3: Support selection by number OR name
parts = [s.strip() for s in raw.split(',')]
selected = []
for p in parts:
    if p.isdigit():
        idx = int(p) - 1
        if idx < 0 or idx >= len(available_classes):
            raise IndexError(f'Number {p} is out of range. Choose between 1 and {len(available_classes)}.')
        selected.append(available_classes[idx])
    else:
        if p not in available_classes:
            raise ValueError(
                f'Class "{p}" not found.\n'
                f'Available: {available_classes}')
        selected.append(p)

if len(selected) < 2:
    raise ValueError(f'At least 2 classes are required for comparison. You selected: {selected}')

print(f'\nSelected: {selected}')

## Cell 7 — Configure analysis options

In [ ]:
# Fix #10: Configurable p-value threshold
print('P-value threshold for significance (default: 0.05):')
p_thresh_input = input('  Enter threshold [0.05]: ').strip()
if p_thresh_input == '':
    P_THRESHOLD = 0.05
else:
    try:
        P_THRESHOLD = float(p_thresh_input)
        if not (0 < P_THRESHOLD < 1):
            raise ValueError
    except ValueError:
        raise ValueError(f'"{p_thresh_input}" is not a valid p-value. Enter a number between 0 and 1 (e.g. 0.01).')
print(f'\nUsing p-value threshold: {P_THRESHOLD}')

# Fix #5: Figure selection
print('\n--- Figure selection ---')
print('Available figures:')
for i, f in enumerate(ALL_FIGURES, 1):
    print(f'  [{i:2d}] {f}')
print('\nEnter figure numbers to generate, separated by commas.')
print('Leave blank to generate ALL figures.')
fig_input = input('Figures [all]: ').strip()
if fig_input == '':
    FIGURES_TO_RUN = None
    print('\n→ Generating all figures.')
else:
    fig_nums = [s.strip() for s in fig_input.split(',')]
    FIGURES_TO_RUN = []
    for n in fig_nums:
        if not n.isdigit() or int(n) < 1 or int(n) > len(ALL_FIGURES):
            raise ValueError(f'"{n}" is not a valid figure number. Choose between 1 and {len(ALL_FIGURES)}.')
        FIGURES_TO_RUN.append(ALL_FIGURES[int(n)-1])
    print(f'\n→ Generating: {FIGURES_TO_RUN}')

## Cell 8 — Run the full analysis
Figures will display inline below as they are generated.

In [ ]:
tag     = '_vs_'.join(s.replace(' ', '') for s in selected)
out_dir = os.path.join(os.path.dirname(filepath), f'results_{tag}_{norm_name}')

# Fix #9: Warn before overwriting existing results
if os.path.isdir(out_dir) and os.listdir(out_dir):
    print(f'⚠  Output folder already exists and contains files:\n   {out_dir}')
    confirm = input('   Overwrite? (yes / no): ').strip().lower()
    if confirm not in ('yes', 'y'):
        raise RuntimeError('Run cancelled to avoid overwriting existing results.\n'
                           'Change your class selection or normalisation to use a different output folder.')
    print('   → Overwriting...')

os.makedirs(out_dir, exist_ok=True)
print(f'Results folder: {out_dir}\n')

results_summary = run_analysis(
    wn, Xn, classes, selected, norm_name, out_dir, hypothesis,
    figures_to_run=FIGURES_TO_RUN,
    p_threshold=P_THRESHOLD,
)

## Cell 9 — AI Hypothesis Interpretation

Uses **Claude claude-opus-4-8** with live web search to relate your spectral results to your hypothesis,
citing recent research on EV Raman spectroscopy.

> **Requires:** `ANTHROPIC_API_KEY` set in Colab Secrets (left panel → 🔑 Secrets → Add) **or** entered below.

In [ ]:
import anthropic
import json
from IPython.display import Markdown, display

# ── API key resolution ───────────────────────────────────────────────────────
api_key = os.environ.get('ANTHROPIC_API_KEY', '')
if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get('ANTHROPIC_API_KEY')
    except Exception:
        pass
if not api_key:
    import getpass
    api_key = getpass.getpass('Enter your Anthropic API key: ')
if not api_key:
    raise RuntimeError('No API key found. Add ANTHROPIC_API_KEY to Colab Secrets or enter it above.')

client = anthropic.Anthropic(api_key=api_key)

# ── Build context from analysis results ─────────────────────────────────────
top_wn_str = ', '.join(f'{w} cm⁻¹ (p={p:.2e})' for w, p in results_summary['top_wavenumbers'])

prompt = f"""You are an expert spectroscopist and cell biologist specialising in extracellular vesicle (EV) Raman spectroscopy.

## Experimental Hypothesis
{hypothesis}

## Analysis Results
- Normalisation method : {norm_name.upper()}
- Classes compared     : {', '.join(selected)}
- LDA cross-validated accuracy: {results_summary['lda_acc']:.1%}
- Significant wavenumbers at p<{P_THRESHOLD}: {results_summary['n_sig']} / {results_summary['n_total']} ({results_summary['pct_sig']}%)
- Top 10 most significant wavenumbers: {top_wn_str}

## Your Task
1. **Interpret the spectral results in relation to the hypothesis.** Explain whether the data supports, partially supports, or contradicts the hypothesis, and why.
2. **Annotate the top significant wavenumbers** — what biomolecules do they correspond to (proteins, lipids, nucleic acids, carbohydrates)? Use established Raman band assignments.
3. **Search for and cite the latest research** (2020–2025) on:
   - Raman spectroscopy of extracellular vesicles
   - B-cell derived EVs and tumor EV intercommunication
   - Spectral biomarkers of immune vesicle reprogramming by tumour microenvironment
4. **Suggest mechanistic explanations** for the observed spectral shifts between classes (e.g., lipid remodelling, nucleic acid cargo transfer, glycoprotein changes).
5. **Propose follow-up experiments** that would strengthen or refute the hypothesis based on these spectral findings.

Format your response as a structured scientific commentary with clear headings. Cite all references with authors, journal, year, and DOI where available.
"""

print('🔍 Searching latest research and generating interpretation...\n')
print('─' * 70)

full_response = ''

with client.messages.stream(
    model='claude-opus-4-8',
    max_tokens=16000,
    thinking={'type': 'adaptive'},
    tools=[{'type': 'web_search_20260209', 'name': 'web_search', 'max_uses': 8}],
    messages=[{'role': 'user', 'content': prompt}],
    betas=['interleaved-thinking-2025-05-14'],
) as stream:
    for event in stream:
        if hasattr(event, 'type'):
            # Stream text blocks live
            if event.type == 'content_block_delta':
                if hasattr(event.delta, 'text'):
                    print(event.delta.text, end='', flush=True)
                    full_response += event.delta.text
            # Show search queries so the user knows what's being looked up
            elif event.type == 'content_block_start':
                block = event.content_block
                if hasattr(block, 'type') and block.type == 'tool_use' and block.name == 'web_search':
                    query = getattr(block, 'input', {}).get('query', '')
                    if query:
                        print(f'\n  🔎 Searching: "{query}"\n', flush=True)

print('\n' + '─' * 70)

# Save interpretation to output directory
interp_path = os.path.join(out_dir, 'ai_hypothesis_interpretation.md')
with open(interp_path, 'w') as f:
    f.write(f'# AI Hypothesis Interpretation\n\n')
    f.write(f'**Model:** claude-opus-4-8  \n')
    f.write(f'**Date:** {datetime.now().strftime("%Y-%m-%d %H:%M")}  \n\n')
    f.write(f'---\n\n')
    f.write(full_response)

print(f'\n✅ Interpretation saved to: {interp_path}')

## Cell 10 — Download all results as a ZIP
Run this to download everything (figures, stats, AI interpretation) to your computer.

In [ ]:
import shutil
from google.colab import files

zip_name = f'results_{tag}_{norm_name}'
shutil.make_archive(zip_name, 'zip', out_dir)
files.download(f'{zip_name}.zip')
print(f'✓ Downloaded: {zip_name}.zip')

## Cell 11 — Run another comparison (optional)

Re-run **Cells 6 → 7 → 8 → 9 → 10** with different classes, normalisation, or figure selection.  
No need to reload data — normalisations are cached.